In [1]:
import pandas as pd
from pandas.api.types import CategoricalDtype
import numpy as np
from pathlib import Path
from itertools import groupby
from operator import itemgetter
from CustomFunctions import DetailedBalance, utils

In [2]:
####### load common directories and data
time_interval = 10 #sec/frame
whichpcs = [1,2,8]
basedir = Path('E:/Aaron/Combined_37C_Confocal_PCA_planar_3D/')
datadir = basedir.joinpath('Data_and_Figs')
FullFrame = pd.read_csv(datadir.joinpath('All_Data_with_CGPS_bins.csv'), index_col=0)
centers = pd.read_csv(datadir.joinpath('PC_bin_centers.csv'), index_col=0)
nbins = len(centers.iloc[:,0])
ttot = time_interval * 180
ntrans = 1
bsiter = 3000
center = [9,8] #coordinates for the flux origin for individual cycle of interest

In [3]:
### restrict data to RANDOM
treatments = ['Random']

savedir = basedir.joinpath('random')
if not savedir.exists():
    savedir.mkdir()

#restrict dataframe to only random experiments
TotalFrame = FullFrame[FullFrame.Treatment=='Random'].copy()

In [3]:
# ### restrict data to PARANITROBLEBBISTATIN
# treatments = ['DMSO','Para-Nitro-Blebbistatin']

# savedir = basedir + 'Para-Nitro-Blebbistatin/'
# if not os.path.exists(savedir):
#     os.makedirs(savedir)

# #limit data to the Para-Nitro-Blebbistatin experiments
# TotalFrame = FullFrame[FullFrame.Experiment == 'Drug'].copy()
# dates = [20240624,20240626,20240701,20241125,20241126,20241127]
# TotalFrame = TotalFrame[TotalFrame.Date.isin(dates)]
# TotalFrame['Treatment'] = pd.Categorical(TotalFrame.Treatment.to_list(), categories=treatments, ordered=True)

In [7]:
# ### restrict data to CK666
# treatments = ['DMSO','CK666']

# savedir = basedir + 'CK666/'
# if not os.path.exists(savedir):
#     os.makedirs(savedir)

# #limit data to the CK666 experiments
# TotalFrame = FullFrame[FullFrame.Experiment == 'Drug'].copy()
# dates = [20240610,20240617,20240620,20241205,20241209]
# TotalFrame = TotalFrame[TotalFrame.Date.isin(dates)]
# TotalFrame['Treatment'] = pd.Categorical(TotalFrame.Treatment.to_list(), categories=treatments, ordered=True)

In [14]:
########### all drugs together
treatments = ['DMSO','CK666','Para-Nitro-Blebbistatin']

savedir = basedir.joinpath('drug')
if not savedir.exists():
    savedir.mkdir()

#limit data to the CK666 experiments
TotalFrame = FullFrame[FullFrame.Treatment.isin(treatments)].copy()
TotalFrame['Treatment'] = pd.Categorical(TotalFrame.Treatment.to_list(), categories=treatments, ordered=True)


In [17]:
### restrict data to galvanotaxis experiments
savedir = basedir.joinpath('galv/')
if not savedir.exists():
    savedir.mkdir()

#restrict dataframe to only random experiments
TotalFrame = FullFrame[FullFrame.Treatment == 'Galvanotaxis'].copy()

In [4]:
if __name__ ==  '__main__':
    ########### get raw transitions and pairs ###########
    rawtrans = DetailedBalance.get_raw_cgps_trajectories(
            TotalFrame, #pandas dataframe with all of the cgps binned data
            whichpcs, #which two PCs to use in the cgps [x,y]
            time_interval, #real time between datapoints
            savedir, #where to save the aggregated trajectories
            )


    ########### interpolate all transitions so that only individual transitions are made ###########
    transdf_sep = DetailedBalance.get_interpolated_cgps_trajectories(
            rawtrans, #pandas dataframe with all of the cgps binned data
            whichpcs, #which two PCs to use in the cgps [x,y]
            savedir, #where to save the aggregated trajectories
            )
    
    
    ############## get the counts of cells leaving 
    trans_rate_df_sep = DetailedBalance.aggregate_transition_counts(
            transdf_sep, #transdf_sep from get_interpolated_cgps_trajectories
            whichpcs, #which two PCs to use in the cgps [x,y]
            savedir, #where to save the aggregated counts
            nbins, #how many bins in the x and y cgps axes
            )

    ############## BOOTSTRAP MANY TRAJECTORIES ##########
    bstrans, bsint, bsframe_sep_full = DetailedBalance.get_bootstrapped_cgps_trajectories(
            rawtrans, #raw transition pairs from get_raw_cgps_trajectories
            whichpcs, #which two PCs to use in the cgps [x,y]
            time_interval, #real time between datapoints
            savedir, #where to save the aggregated counts
            nbins, #how many bins in the x and y cgps axes
            ttot, #set the total bootstrap time
            ntrans, #how many transitions to sample at each step
            bsiter, #number of times to bootstrap
            )


#     ############# open average bootstrapped currents ###################
#     bsfield_sep = DetailedBalance.get_avg_current_error(
#             bsframe_sep_full, #transition rates in the cgps from get_bootstrapped_cgps_trajectories
#             whichpcs, #which two PCs to use in the cgps [x,y]
#             savedir, #where to save the aggregated counts
#             nbins, #how many bins in the x and y cgps axes
#             ntrans, #how many transitions to sample at each step
#             )
    

Aggregated transitions
Finished interpolating trajectories
Total time observed in this CGPS was 1350.9217186272904 minutes
Finished finding transition rates
Boostrapping trajectories with 1 transition samples for Random


100%|██████████| 3000/3000 [01:52<00:00, 26.57it/s]


Interpolating trajectories for Random


100%|██████████| 3000/3000 [00:09<00:00, 322.50it/s]


Calculating bootstrapped CGPS transition rates for Random


100%|██████████| 3000/3000 [36:19<00:00,  1.38it/s]  


ValueError: Length of values (675000) does not match length of index (10125000)

In [13]:
################ get bootstrapped aer and cfs ##################
if __name__ ==  '__main__':
    bstranspath = savedir.joinpath(f'PC{whichpcs[0]}-PC{whichpcs[1]}_bootstrapped_{ntrans}_transitions.csv')
    if bstranspath.exists():
        bstrans = pd.read_csv(bstranspath, index_col=0)
        #get the area scaling in x and y based on the size of the bins in the cgps
        xyscaling = [centers[f'PC{whichpcs[0]}'].diff().mean(),centers[f'PC{whichpcs[1]}'].diff().mean()]

        DetailedBalance.get_aer_cf(
            bstrans, #boostrapped transitions from get_bootstrapped_cgps_trajectories
            nbins, #how many bins in the x and y cgps axes
            xyscaling, #scaling of the bins in real units of whatever the CGPS axis parameters are
            center, #origin in [x bin,y bin]
            savedir, #where to save calculated aers and cfs
            whichpcs, #which two PCs to use in the cgps [x,y]
            ntrans, #how many transitions to sample at each step
            )

100%|██████████| 3000/3000 [00:06<00:00, 441.12it/s]


In [14]:
########## get individual cell actual aer and cfs ###############


if __name__ ==  '__main__':
    rawtranspath = savedir.joinpath(f'PC{whichpcs[0]}-PC{whichpcs[1]}_transitions_separated.csv')
    if rawtranspath.exists():
        #open the raw transitions in case I didn't just generate them
        rawtrans = pd.read_csv(rawtranspath, index_col = 0)
        #get the area scaling in x and y based on the size of the bins in the cgps
        xyscaling = [centers[f'PC{whichpcs[0]}'].diff().mean(),centers[f'PC{whichpcs[1]}'].diff().mean()]

        results = []
        for i, cells in rawtrans.groupby('CellID'):
            cells, runs = utils.get_consecutive_timepoints(cells, 'frame',1)
            for r in runs:
                cell = cells.iloc[r].reset_index(drop=True)
                results.append(DetailedBalance.get_area_enclosing_rate((
                    cell,
                    nbins,
                    xyscaling,
                    center,
                    )))

        #make a dataframe and save it
        allaers = pd.concat(results).reset_index(drop=True)
        justaers = allaers[['CellID','cell','Treatment','aer','angular_velocity']].copy()
        justaers.to_csv(savedir.joinpath(f'PC{whichpcs[0]}-PC{whichpcs[1]}_raw_transition_aer_cf.csv'))


In [4]:
############# create all CGPSs #############

binlist = [i for i in TotalFrame.columns.to_list() if 'bin' in i]
allsavedir = savedir.joinpath('allCGPS')
if not allsavedir.exists():
    allsavedir.mkdir(parents = True)
for a in range(1,len(binlist)+1):
    for b in range(1,len(binlist)+1):
        if a == b:
            continue
        elif allsavedir.joinpath(f'PC{b}-PC{a}_interpolated_transitions_separated.csv').exists():
            print('Already made this CGPS')
            continue
        else:
            #set the PCs
            abwhichpcs = [a,b]
            if __name__ ==  '__main__':
                
                ########### get raw transitions and pairs ###########
                rawtrans = DetailedBalance.get_raw_cgps_trajectories(
                        TotalFrame, #pandas dataframe with all of the cgps binned data
                        abwhichpcs, #which two PCs to use in the cgps [x,y]
                        time_interval, #real time between datapoints
                        allsavedir, #where to save the aggregated trajectories
                        )
                
                ########### interpolate all transitions so that only individual transitions are made ###########
                transdf_sep = DetailedBalance.get_interpolated_cgps_trajectories(
                        rawtrans, #pandas dataframe with all of the cgps binned data
                        abwhichpcs, #which two PCs to use in the cgps [x,y]
                        time_interval, #real time between datapoints
                        allsavedir, #where to save the aggregated trajectories
                        )

                ############## get the counts of cells leaving 
                trans_rate_df_sep = DetailedBalance.aggregate_transition_counts(
                        transdf_sep, #transdf_sep from get_interpolated_cgps_trajectories
                        abwhichpcs, #which two PCs to use in the cgps [x,y]
                        allsavedir, #where to save the aggregated counts
                        nbins, #how many bins in the x and y cgps axes
                        )
                
                
            

Aggregated transitions
Finished interpolating trajectories
Total time observed in this CGPS was 1853.1585572470565 minutes
Finished finding transition rates
Aggregated transitions
Finished interpolating trajectories
Total time observed in this CGPS was 1849.7554358140917 minutes
Finished finding transition rates
Aggregated transitions
Finished interpolating trajectories
Total time observed in this CGPS was 1851.971505260773 minutes
Finished finding transition rates
Aggregated transitions
Finished interpolating trajectories
Total time observed in this CGPS was 1851.926469078032 minutes
Finished finding transition rates
Aggregated transitions
Finished interpolating trajectories
Total time observed in this CGPS was 1848.376407254082 minutes
Finished finding transition rates
Aggregated transitions
Finished interpolating trajectories
Total time observed in this CGPS was 1848.7841454397073 minutes
Finished finding transition rates
Aggregated transitions
Finished interpolating trajectories
To

In [5]:
########### calculate all the aers and cfs around all the pairwise cgps ###############

binlist = [i for i in TotalFrame.columns.to_list() if 'bin' in i]
allsavedir = savedir.joinpath('allCGPS')
if not allsavedir.exists():
    allsavedir.mkdir(parents = True)
    
#### all the cgps origins determinned by visual inspection (specifically for random treatment)
allorigins = [[[8,9],[8,8],[9,7],[8,8],[9,9],[8,8],[9,8]],
                [[8,9],[9,7],[8,8],[9,9],[8,8],[9,7]],
                    [[8,8],[9,8],[8,7],[8,8],[6,8]],
                        [[7,8],[7,8],[8,8],[8,7]],
                            [[8,8],[8,8],[8,8]],
                                [[8,8],[8,8]],
                                    [[7,7]]]
    
for a in range(1,len(binlist)+1):
    for b in range(1,len(binlist)+1):
        if a == b:
            continue
        elif allsavedir.joinpath(f'PC{b}-PC{a}_interpolated_transitions_separated.csv').exists():
            print('Already made this plot')
            continue
        else:
            #set the PCs
            abwhichpcs = [a,b]
            if __name__ ==  '__main__':
                
                #### open the transitions
                rawtrans = pd.read_csv(allsavedir.joinpath(
                    f'PC{abwhichpcs[0]}-PC{abwhichpcs[1]}_transitions_separated.csv'), index_col=0)

                
                ############## BOOTSTRAP MANY TRAJECTORIES ##########
                bstrans, bsint, bsframe_sep_full = DetailedBalance.get_bootstrapped_cgps_trajectories(
                        rawtrans, #raw transition pairs from get_raw_cgps_trajectories
                        abwhichpcs, #which two PCs to use in the cgps [x,y]
                        time_interval, #real time between datapoints
                        allsavedir, #where to save the aggregated counts
                        nbins, #how many bins in the x and y cgps axes
                        ttot, #set the total bootstrap time
                        ntrans, #how many transitions to sample at each step
                        bsiter = 3000, #number of times to bootstrap
                        )


                ############# open average bootstrapped currents ###################
                bsfield_sep = DetailedBalance.get_avg_current_error(
                        bsframe_sep_full, #transition rates in the cgps from get_bootstrapped_cgps_trajectories
                        abwhichpcs, #which two PCs to use in the cgps [x,y]
                        allsavedir, #where to save the aggregated counts
                        nbins, #how many bins in the x and y cgps axes
                        ntrans, #how many transitions to sample at each step
                        )
    
                
                
                ############# measure aer and cycling frequencies ###########
                #add specific scaling
                xyscaling = [centers[f'PC{abwhichpcs[0]}'].diff().mean(),centers[f'PC{abwhichpcs[1]}'].diff().mean()]
                #set the origin to the actual center
                center = allorigins[int(a-1)][int(b-(2+a-1))]

                DetailedBalance.get_aer_cf(
                    bstrans,
                    nbins, #how many bins in the x and y cgps axes
                    xyscaling, #scaling of the bins in real units of whatever the CGPS axis parameters are
                    center, #origin in [x bin,y bin]
                    allsavedir, #where to save calculated aers and cfs
                    abwhichpcs, #which two PCs to use in the cgps [x,y]
                    ntrans, #how many transitions to sample at each step
                    )

Boostrapping trajectories with 1 transition samples for Random


100%|██████████| 3000/3000 [01:03<00:00, 47.04it/s]


Interpolating trajectories for Random
Calculating bootstrapped CGPS transition rates for Random


100%|██████████| 3000/3000 [00:55<00:00, 53.66it/s]


Finished bootstrapping


...

100%|██████████| 3000/3000 [00:56<00:00, 52.68it/s]


Finished bootstrapping


100%|██████████| 3000/3000 [00:06<00:00, 444.41it/s]


Boostrapping trajectories with 1 transition samples for Random


 75%|███████▍  | 2238/3000 [00:50<00:15, 49.47it/s]IOPub message rate exceeded.
The notebook server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--NotebookApp.iopub_msg_rate_limit`.

Current values:
NotebookApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
NotebookApp.rate_limit_window=3.0 (secs)

